In [ ]:
%reset -f

In [ ]:
import pandas as pd
import random

In [ ]:
def unzip(zipped_list: list[tuple], output_length: int = 1) -> tuple:
    if len(zipped_list) == 0:
        return tuple([] for _ in range(output_length))

    return tuple(map(list, zip(*zipped_list, strict=True)))

print(unzip([(1,2), (3,4), (5,6)]))

In [ ]:
def split_evenely(items: list, classifier_fn):
    partition: dict[int, list] = {}

    # Partition items based on the classifications
    for item in items:
        key = classifier_fn(item)
        if key in partition:
            partition[key].append(item)
        else:
            partition[key] = [item]

    min_count = len(min(partition.values(), key=len))

    # Append the partitioned sublists together up to the min_count
    new_items = []
    for sublist in partition.values():
        new_items += sublist[0:min_count]

    # random.shuffle(new_items)
    return new_items

xs = [-1,1,3,2,1,1,1,2,1,-2,1,2,-3,-3,3,2,2,1,3]
print(split_evenely(xs, lambda x: x*x))

In [ ]:
def split_df_evenly(df: pd.DataFrame, classifier_fn):
    return pd.DataFrame(
        split_evenely((x[1] for x in df.iterrows()), classifier_fn)
    )

In [ ]:
# df = pd.read_csv("stateData_100000_10.csv")
df = pd.read_csv("stateData_100000_10_test.csv")

In [ ]:
indices = set(df["trialIndex"])

# FIXME: Optimize this to not retrieve whole dataframe subset, just first row of interest
# iter_samples = [df[df.trialIndex == i].iloc[0] for i in indices]
# iter_samples = [
#     (query := df[df.trialIndex == i]).iloc[1 % len(query)]
#     for i in indices
# ]
iter_samples = [
    (query := df[df.trialIndex == i]).iloc[2 % len(query)]
    for i in indices
]
# iter_samples = [df[df.trialIndex == i].iloc[-1] for i in indices]
iter_sample_df = pd.DataFrame(iter_samples)
iter_sample_df

In [ ]:
bal_df = split_df_evenly(iter_sample_df, lambda row: row["converges"])
bal_df

In [ ]:
bal_df = bal_df.sample(frac=1).reset_index(drop=True)
display(bal_df)

bal_df.to_csv("stateData_2000_10_itr2_test.csv", index=False)